# EDA com Full Join das Abas

Este notebook carrega o Excel `BASE DE DADOS PEDE 2024 - DATATHON.xlsx`, faz um *full join* entre as abas `PEDE2022`, `PEDE2023`, `PEDE2024` pela chave `RA` e gera um EDA básico do resultado.

In [13]:
import pandas as pd

# ============================================================================
# CONFIGURAÇÕES
# ============================================================================

FILE_PATH = "/workspaces/Datathon-Machine-Learning-Engineering/data/BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

# ============================================================================
# 1. CARREGAR DADOS
# ============================================================================

print("📂 Carregando dados...")
df_2022 = pd.read_excel(FILE_PATH, sheet_name="PEDE2022")
df_2023 = pd.read_excel(FILE_PATH, sheet_name="PEDE2023")
df_2024 = pd.read_excel(FILE_PATH, sheet_name="PEDE2024")

print(f"  ✓ 2022: {df_2022.shape[0]:,} linhas x {df_2022.shape[1]} colunas")
print(f"  ✓ 2023: {df_2023.shape[0]:,} linhas x {df_2023.shape[1]} colunas")
print(f"  ✓ 2024: {df_2024.shape[0]:,} linhas x {df_2024.shape[1]} colunas")

📂 Carregando dados...


  ✓ 2022: 860 linhas x 42 colunas
  ✓ 2023: 1,014 linhas x 48 colunas
  ✓ 2024: 1,156 linhas x 50 colunas


In [14]:
COLUNAS_SEM_SUFIXO = ['RA']

# Função para adicionar sufixo nas colunas
def add_suffix_to_columns(df, suffix, exclude_cols):
    """
    Adiciona sufixo às colunas, exceto as que estão na lista de exclusão
    """
    new_columns = {}
    for col in df.columns:
        if col in exclude_cols:
            new_columns[col] = col  # Mantém o nome original
        else:
            new_columns[col] = f"{col}_{suffix}"  # Adiciona sufixo
    
    return df.rename(columns=new_columns)

# Renomeia cada base
df_2022_renamed = add_suffix_to_columns(df_2022, '2022', COLUNAS_SEM_SUFIXO)
df_2023_renamed = add_suffix_to_columns(df_2023, '2023', COLUNAS_SEM_SUFIXO)
df_2024_renamed = add_suffix_to_columns(df_2024, '2024', COLUNAS_SEM_SUFIXO)

print("  ✓ Colunas renomeadas")

  ✓ Colunas renomeadas


In [15]:

# Primeiro merge: 2022 + 2023
merged = df_2022_renamed.merge(
    df_2023_renamed,
    on='RA',
    how='outer'  # Full join - mantém todas as linhas
)


# Segundo merge: (2022+2023) + 2024
merged = merged.merge(
    df_2024_renamed,
    on='RA',
    how='outer'  # Full join - mantém todas as linhas
)

print(merged.shape)


(1661, 138)


In [16]:
total_alunos = merged['RA'].nunique()
print(f"  • Total de alunos únicos (RA): {total_alunos:,}")

# Alunos por ano
alunos_2022 = df_2022_renamed['RA'].nunique()
alunos_2023 = df_2023_renamed['RA'].nunique()
alunos_2024 = df_2024_renamed['RA'].nunique()

print(f"\n  • Alunos em 2022: {alunos_2022:,}")
print(f"  • Alunos em 2023: {alunos_2023:,}")
print(f"  • Alunos em 2024: {alunos_2024:,}")

# Alunos que aparecem em todos os anos
alunos_2022_2023 = set(df_2022_renamed['RA']) & set(df_2023_renamed['RA'])
alunos_2023_2024 = set(df_2023_renamed['RA']) & set(df_2024_renamed['RA'])
alunos_todos_anos = alunos_2022_2023 & set(df_2024_renamed['RA'])

print(f"\n  • Alunos em 2022 E 2023: {len(alunos_2022_2023):,}")
print(f"  • Alunos em 2023 E 2024: {len(alunos_2023_2024):,}")
print(f"  • Alunos todos anos: {len(alunos_todos_anos):,}")


  • Total de alunos únicos (RA): 1,661

  • Alunos em 2022: 860
  • Alunos em 2023: 1,014
  • Alunos em 2024: 1,156

  • Alunos em 2022 E 2023: 600
  • Alunos em 2023 E 2024: 765
  • Alunos todos anos: 468


In [17]:
print(merged.head())

print("\n📊 Info do dataset:")
print(f"  • Shape: {merged.shape}")
print(f"  • Colunas: {merged.shape[1]}")
print(f"  • Memória: {merged.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

        RA  Fase_2022 Turma_2022  Nome_2022  Ano nasc_2022  Idade 22_2022  \
0     RA-1        7.0          A    Aluno-1         2003.0           19.0   
1    RA-10        7.0          A   Aluno-10         2004.0           18.0   
2   RA-100        4.0          A  Aluno-100         2009.0           13.0   
3  RA-1000        NaN        NaN        NaN            NaN            NaN   
4  RA-1001        NaN        NaN        NaN            NaN            NaN   

  Gênero_2022  Ano ingresso_2022 Instituição de ensino_2022 Pedra 20_2022  \
0      Menina             2016.0             Escola Pública      Ametista   
1      Menina             2021.0             Escola Pública           NaN   
2      Menina             2019.0               Rede Decisão      Ametista   
3         NaN                NaN                        NaN           NaN   
4         NaN                NaN                        NaN           NaN   

   ... IPV_2024 IAN_2024          Fase Ideal_2024  Defasagem_2024  \
0  ..

In [18]:
# Valores unicos por coluna (brutos)
cols_fase = ["Fase_2022", "Fase_2023", "Fase_2024"]
for col in cols_fase:
    if col not in merged.columns:
        print(f"Coluna ausente: {col}")
        continue
    valores = merged[col].dropna().astype("string")
    unicos = sorted(valores.unique())
    print(f"\n{col} -> {len(unicos)} valores unicos (bruto):")
    print(unicos)



Fase_2022 -> 8 valores unicos (bruto):
['0.0', '1.0', '2.0', '3.0', '4.0', '5.0', '6.0', '7.0']

Fase_2023 -> 9 valores unicos (bruto):
['ALFA', 'FASE 1', 'FASE 2', 'FASE 3', 'FASE 4', 'FASE 5', 'FASE 6', 'FASE 7', 'FASE 8']

Fase_2024 -> 72 valores unicos (bruto):
['1A', '1B', '1C', '1D', '1E', '1G', '1H', '1J', '1K', '1L', '1M', '1N', '1P', '1R', '2A', '2B', '2C', '2D', '2G', '2H', '2I', '2K', '2L', '2M', '2N', '2P', '2R', '2U', '3A', '3B', '3C', '3D', '3F', '3G', '3H', '3I', '3K', '3L', '3M', '3N', '3P', '3R', '3U', '4A', '4B', '4C', '4F', '4H', '4L', '4M', '4N', '4R', '5A', '5B', '5C', '5D', '5F', '5G', '5L', '5M', '5N', '6A', '6L', '7A', '7E', '8A', '8B', '8D', '8E', '8F', '9', 'ALFA']


In [21]:
# Padronizacao das fases (simples, coluna por coluna)
FASE_MAP = {
    "0": "ALFA",
    "1": "FASE 1",
    "2": "FASE 2",
    "3": "FASE 3",
    "4": "FASE 4",
    "5": "FASE 5",
    "6": "FASE 6",
    "7": "FASE 7",
    "8": "FASE 8",
}

# Fase 2022
fase_2022_base = merged["Fase_2022"].astype("string").str.strip().str.upper()
fase_2022_digit = fase_2022_base.str.extract(r"(\d)", expand=False)
merged["Fase_2022_adj"] = fase_2022_digit.map(FASE_MAP).fillna(fase_2022_base)

# Fase 2023 (ja correta, mas padroniza caixa/espacos)
fase_2023_base = merged["Fase_2023"].astype("string").str.strip().str.upper()
fase_2023_digit = fase_2023_base.str.extract(r"(\d)", expand=False)
merged["Fase_2023_adj"] = fase_2023_digit.map(FASE_MAP).fillna(fase_2023_base)

# Fase 2024 (mistura numero e texto -> pega o primeiro digito)
fase_2024_base = merged["Fase_2024"].astype("string").str.strip().str.upper()
fase_2024_digit = fase_2024_base.str.extract(r"(\d)", expand=False)
merged["Fase_2024_adj"] = fase_2024_digit.map(FASE_MAP).fillna(fase_2024_base)

print(
    merged[[
        "Fase_2022",
        "Fase_2022_adj",
        "Fase_2023",
        "Fase_2023_adj",
        "Fase_2024",
        "Fase_2024_adj",
    ]].head()
)


   Fase_2022 Fase_2022_adj Fase_2023 Fase_2023_adj Fase_2024 Fase_2024_adj
0        7.0        FASE 7    FASE 8        FASE 8        8E        FASE 8
1        7.0        FASE 7       NaN           NaN       NaN           NaN
2        4.0        FASE 4       NaN           NaN       NaN           NaN
3        NaN           NaN      ALFA          ALFA        1N        FASE 1
4        NaN           NaN      ALFA          ALFA        1N        FASE 1


In [22]:
# Valores unicos por coluna (brutos)
cols_fase = ["Fase_2022", "Fase_2023", "Fase_2024","Fase_2022_adj", "Fase_2023_adj", "Fase_2024_adj"]
for col in cols_fase:
    if col not in merged.columns:
        print(f"Coluna ausente: {col}")
        continue
    valores = merged[col].dropna().astype("string")
    unicos = sorted(valores.unique())
    print(f"\n{col} -> {len(unicos)} valores unicos (bruto):")
    print(unicos)


Fase_2022 -> 8 valores unicos (bruto):
['0.0', '1.0', '2.0', '3.0', '4.0', '5.0', '6.0', '7.0']

Fase_2023 -> 9 valores unicos (bruto):
['ALFA', 'FASE 1', 'FASE 2', 'FASE 3', 'FASE 4', 'FASE 5', 'FASE 6', 'FASE 7', 'FASE 8']

Fase_2024 -> 72 valores unicos (bruto):
['1A', '1B', '1C', '1D', '1E', '1G', '1H', '1J', '1K', '1L', '1M', '1N', '1P', '1R', '2A', '2B', '2C', '2D', '2G', '2H', '2I', '2K', '2L', '2M', '2N', '2P', '2R', '2U', '3A', '3B', '3C', '3D', '3F', '3G', '3H', '3I', '3K', '3L', '3M', '3N', '3P', '3R', '3U', '4A', '4B', '4C', '4F', '4H', '4L', '4M', '4N', '4R', '5A', '5B', '5C', '5D', '5F', '5G', '5L', '5M', '5N', '6A', '6L', '7A', '7E', '8A', '8B', '8D', '8E', '8F', '9', 'ALFA']

Fase_2022_adj -> 8 valores unicos (bruto):
['ALFA', 'FASE 1', 'FASE 2', 'FASE 3', 'FASE 4', 'FASE 5', 'FASE 6', 'FASE 7']

Fase_2023_adj -> 9 valores unicos (bruto):
['ALFA', 'FASE 1', 'FASE 2', 'FASE 3', 'FASE 4', 'FASE 5', 'FASE 6', 'FASE 7', 'FASE 8']

Fase_2024_adj -> 10 valores unicos (bruto)